In [1]:

# Step 1: Load and explore the dataset

from sklearn.datasets import fetch_20newsgroups
import pandas as pd

# Load the training and testing documents
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)

# Convert the dataset into a DataFrame
df = pd.DataFrame({
    "document_id": range(len(dataset.data)),
    "category_id": dataset.target,
    "category": [dataset.target_names[i] for i in dataset.target],
    "text": dataset.data
})

# Remove empty documents
df["text"] = df["text"].fillna("").astype(str)
df = df[df["text"].str.strip().ne("")].reset_index(drop=True)

# Display dataset information
print("Dataset loaded successfully!")
print("Total documents:", len(df))
print("Total categories:", df["category"].nunique())

print("\nCategory names:")
print(dataset.target_names)

print("\nFirst 5 documents:")
display(df.head())

print("\nDocuments per category:")
display(df["category"].value_counts().to_frame("document_count"))


Dataset loaded successfully!
Total documents: 18331
Total categories: 20

Category names:
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

First 5 documents:


,document_id,category_id,category,text
0,0,10,rec.sport.hockey,\n\nI am sure some bashers of Pens fans are pr...
1,1,3,comp.sys.ibm.pc.hardware,My brother is in the market for a high-perform...
2,2,17,talk.politics.mideast,\n\n\n\n\tFinally you said what you dream abou...
3,3,3,comp.sys.ibm.pc.hardware,\nThink!\n\nIt's the SCSI card doing the DMA t...
4,4,4,comp.sys.mac.hardware,1) I have an old Jasmine drive which I cann...



Documents per category:


,document_count
category,
comp.windows.x,982
rec.sport.hockey,975
soc.religion.christian,975
rec.motorcycles,969
comp.sys.ibm.pc.hardware,964
sci.crypt,962
sci.med,960
misc.forsale,959
sci.electronics,958


In [2]:

import re
import pandas as pd

# 1. Make a copy of the original dataset
clean_df = df.copy()

# 2. Clean whitespace and unwanted control characters
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    return text.strip()

clean_df["text"] = clean_df["text"].apply(clean_text)

# 3. Remove empty documents
clean_df = clean_df[clean_df["text"].str.len() > 0]

# 4. Remove duplicate documents
clean_df = clean_df.drop_duplicates(subset=["text"])

# 5. Remove very short documents (fewer than 30 characters)
clean_df = clean_df[clean_df["text"].str.len() >= 30]

# 6. Reset document IDs after cleaning
clean_df = clean_df.reset_index(drop=True)
clean_df["document_id"] = range(len(clean_df))

# 7. Create normalized text for keyword matching
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_df["normalized_text"] = clean_df["text"].apply(normalize_text)

# Display results
print("Original documents:", len(df))
print("Documents after cleaning:", len(clean_df))
print("Documents removed:", len(df) - len(clean_df))
print("\nSample cleaned documents:")

display(clean_df[["document_id", "category", "text"]].head())

# Check for remaining empty documents and duplicates
print("\nEmpty documents:", (clean_df["text"].str.len() == 0).sum())
print("Duplicate documents:", clean_df["text"].duplicated().sum())


Original documents: 18331
Documents after cleaning: 18059
Documents removed: 272

Sample cleaned documents:


,document_id,category,text
0,0,rec.sport.hockey,I am sure some bashers of Pens fans are pretty...
1,1,comp.sys.ibm.pc.hardware,My brother is in the market for a high-perform...
2,2,talk.politics.mideast,Finally you said what you dream about. Mediter...
3,3,comp.sys.ibm.pc.hardware,Think! It's the SCSI card doing the DMA transf...
4,4,comp.sys.mac.hardware,1) I have an old Jasmine drive which I cannot ...



Empty documents: 0
Duplicate documents: 0


In [3]:

# Split documents into overlapping chunks

CHUNK_SIZE = 150
CHUNK_OVERLAP = 30

chunk_records = []

for _, row in clean_df.iterrows():
    words = row["text"].split()

    start = 0
    chunk_index = 0

    while start < len(words):
        end = min(start + CHUNK_SIZE, len(words))
        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunk_records.append({
                "chunk_id": len(chunk_records),
                "document_id": row["document_id"],
                "category": row["category"],
                "chunk_index": chunk_index,
                "text": chunk_text,
                "normalized_text": normalize_text(chunk_text)
            })

        if end == len(words):
            break

        start += CHUNK_SIZE - CHUNK_OVERLAP
        chunk_index += 1

chunks_df = pd.DataFrame(chunk_records)

print("Total cleaned documents:", len(clean_df))
print("Total text chunks:", len(chunks_df))
print("\nSample chunks:")
display(chunks_df.head(5))


Total cleaned documents: 18059
Total text chunks: 35842

Sample chunks:


,chunk_id,document_id,category,chunk_index,text,normalized_text
0,0,0,rec.sport.hockey,0,I am sure some bashers of Pens fans are pretty...,i am sure some bashers of pens fans are pretty...
1,1,1,comp.sys.ibm.pc.hardware,0,My brother is in the market for a high-perform...,my brother is in the market for a high perform...
2,2,2,talk.politics.mideast,0,Finally you said what you dream about. Mediter...,finally you said what you dream about mediterr...
3,3,2,talk.politics.mideast,1,"too? What ever you say ""regional killer"", if y...",too what ever you say regional killer if you d...
4,4,3,comp.sys.ibm.pc.hardware,0,Think! It's the SCSI card doing the DMA transf...,think it s the scsi card doing the dma transfe...


In [4]:

import os

os.makedirs("/content/data", exist_ok=True)

clean_df.to_csv(
    "/content/data/cleaned_documents.csv",
    index=False
)

chunks_df.to_csv(
    "/content/data/document_chunks.csv",
    index=False
)

print("Preprocessed data saved successfully!")
print("File 1: /content/data/cleaned_documents.csv")
print("File 2: /content/data/document_chunks.csv")


Preprocessed data saved successfully!
File 1: /content/data/cleaned_documents.csv
File 2: /content/data/document_chunks.csv


In [5]:

!pip -q install sentence-transformers


In [6]:

from sentence_transformers import SentenceTransformer
import numpy as np

# Load a lightweight sentence embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Convert each document chunk into an embedding
chunk_texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Embeddings generated successfully!")
print("Number of chunks:", len(chunk_texts))
print("Embedding matrix shape:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/561 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of chunks: 35842
Embedding matrix shape: (35842, 384)


In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    chunks_df["normalized_text"]
)

print("TF-IDF index created successfully!")
print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF index created successfully!
TF-IDF matrix shape: (35842, 50000)


In [8]:

from sklearn.metrics.pairwise import cosine_similarity

def search_documents(query, top_n=5, semantic_weight=0.6):
    if not query or not query.strip():
        return pd.DataFrame()

    if not 0 <= semantic_weight <= 1:
        raise ValueError("semantic_weight must be between 0 and 1")

    # Keyword search
    query_normalized = normalize_text(query)
    query_tfidf = tfidf_vectorizer.transform([query_normalized])

    keyword_scores = (
        tfidf_matrix @ query_tfidf.T
    ).toarray().ravel()

    # Semantic search
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    semantic_scores = (
        embeddings @ query_embedding[0]
    )

    # Combine both scores
    hybrid_scores = (
        semantic_weight * semantic_scores
        + (1 - semantic_weight) * keyword_scores
    )

    # Return the top results
    top_n = max(1, int(top_n))
    top_indices = np.argsort(hybrid_scores)[::-1][:top_n]

    results = chunks_df.iloc[top_indices][
        ["chunk_id", "document_id", "category", "text"]
    ].copy()

    results["keyword_score"] = keyword_scores[top_indices]
    results["semantic_score"] = semantic_scores[top_indices]
    results["hybrid_score"] = hybrid_scores[top_indices]

    return results.reset_index(drop=True)

print("Hybrid search function created successfully!")


Hybrid search function created successfully!


In [9]:

query = "machine learning algorithms and neural networks"

results = search_documents(query, top_n=5)

print("Search query:", query)

display(
    results[
        [
            "document_id",
            "category",
            "keyword_score",
            "semantic_score",
            "hybrid_score",
            "text"
        ]
    ]
)


Search query: machine learning algorithms and neural networks


,document_id,category,keyword_score,semantic_score,hybrid_score,text
0,16803,sci.med,0.307613,0.501410,0.423891,problems in both academia and industry. 3:00-5...
1,15456,comp.graphics,0.399527,0.413346,0.407818,CALL FOR PAPERS =============== Progress In Ne...
2,15456,comp.graphics,0.260708,0.471126,0.386959,"topics include, but are not limited to: shape ..."
3,18032,alt.atheism,0.372712,0.380494,0.377381,I'm sure there are many people who work with n...
4,16803,sci.med,0.192435,0.435015,0.337983,"Science and Biology at Rutgers, and has extens..."


In [10]:

query = "space exploration and NASA"

results = search_documents(query, top_n=5)

print("Search query:", query)

display(
    results[
        [
            "document_id",
            "category",
            "keyword_score",
            "semantic_score",
            "hybrid_score",
            "text"
        ]
    ]
)


Search query: space exploration and NASA


,document_id,category,keyword_score,semantic_score,hybrid_score,text
0,14514,sci.space,0.367103,0.603353,0.508853,"Indeed, with few notable exceptions, such as V..."
1,13697,sci.space,0.314385,0.631295,0.504531,"We are not at the end of the Space Age, but on..."
2,9207,sci.space,0.258089,0.634057,0.483670,studies in this area of space exploration. I p...
3,9207,sci.space,0.246631,0.595651,0.456043,"and SCIENTIFIC AMERICAN magazines. TIME, NEWSW..."
4,9207,sci.space,0.240170,0.579283,0.443637,"reading the following works, categorized in th..."


In [11]:

query = "computer graphics and image processing"

results = search_documents(query, top_n=5)

print("Search query:", query)

display(
    results[
        [
            "document_id",
            "category",
            "keyword_score",
            "semantic_score",
            "hybrid_score",
            "text"
        ]
    ]
)


Search query: computer graphics and image processing


,document_id,category,keyword_score,semantic_score,hybrid_score,text
0,5558,comp.graphics,0.442118,0.613785,0.545118,"I usually use ""Algorithms for graphics and ima..."
1,3445,comp.graphics,0.318603,0.553580,0.459589,A book that I can somewhat recommend is : Prat...
2,7741,comp.graphics,0.221552,0.584375,0.439246,Technion - Israel Institute of Technology Depa...
3,12613,comp.graphics,0.211235,0.576232,0.430233,Greetings this is a general call for informati...
4,16324,sci.med,0.315869,0.488379,0.419375,"This is not flame, or abuse, nor do I want to ..."


In [12]:

query = "politics and government"

results = search_documents(query, top_n=5)

print("Search query:", query)

display(
    results[
        [
            "document_id",
            "category",
            "keyword_score",
            "semantic_score",
            "hybrid_score",
            "text"
        ]
    ]
)


Search query: politics and government


,document_id,category,keyword_score,semantic_score,hybrid_score,text
0,15154,sci.crypt,0.203248,0.439495,0.344996,>government maintaining a secret of some kind.
1,15922,talk.politics.misc,0.212921,0.420641,0.337553,"in one city only, and if yes, which? Please po..."
2,9045,sci.crypt,0.251230,0.371705,0.323515,"ethics, I will use it anyway as long I think i..."
3,13459,talk.politics.misc,0.194137,0.402742,0.319300,In a debate about the student government here ...
4,2681,rec.motorcycles,0.211987,0.380318,0.312985,I first heard it about academic politics but t...


In [13]:

query = "baseball and sports"

results = search_documents(query, top_n=5)

print("Search query:", query)

display(
    results[
        [
            "document_id",
            "category",
            "keyword_score",
            "semantic_score",
            "hybrid_score",
            "text"
        ]
    ]
)


Search query: baseball and sports


,document_id,category,keyword_score,semantic_score,hybrid_score,text
0,1646,rec.sport.baseball,0.245902,0.497831,0.397059,Me too! I would like any park or action gif or...
1,5159,rec.sport.baseball,0.232742,0.486228,0.384834,in a relief pitcher who then takes (8? 10?) pi...
2,7058,rec.sport.baseball,0.139820,0.531485,0.374819,the owners are whining about baseball not bein...
3,15150,rec.sport.baseball,0.215568,0.478867,0.373547,baseball more appealing to a bigger crowd is g...
4,2807,rec.sport.hockey,0.240101,0.446795,0.364118,"What, did you leave the room each of the 100 o..."


In [18]:

%pip uninstall -y google-genai


Found existing installation: google-genai 2.33.0
Uninstalling google-genai-2.33.0:
  Successfully uninstalled google-genai-2.33.0


In [19]:

%pip install -q "google-auth==2.49.0" requests


In [20]:

import google.auth
import requests

print("Google Auth version:", google.auth.__version__)
print("Requests is working:", requests.__version__)

assert google.auth.__version__ == "2.49.0"
print("Environment check passed!")


Google Auth version: 2.49.0
Requests is working: 2.32.4
Environment check passed!


In [21]:
from getpass import getpass
import requests

api_key = getpass("Enter your Gemini API key: ")

print("API key entered securely.")

Enter your Gemini API key: ··········
API key entered securely.


In [22]:
def generate_summary_with_gemini(prompt):
    url = (
        "https://generativelanguage.googleapis.com/"
        "v1beta/models/gemini-2.5-flash:generateContent"
    )

    response = requests.post(
        url,
        params={"key": api_key},
        headers={"Content-Type": "application/json"},
        json={
            "contents": [
                {
                    "parts": [{"text": prompt}]
                }
            ],
            "generationConfig": {
                "temperature": 0.2
            }
        },
        timeout=60
    )

    if not response.ok:
        raise RuntimeError(
            f"Gemini API error {response.status_code}: "
            f"{response.text[:1000]}"
        )

    data = response.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]


print("Gemini summarization function created successfully!")

Gemini summarization function created successfully!


In [23]:
def summarize_documents(query, summary_length="medium", top_n=5):
    length_instructions = {
        "short": "Write approximately 50 words.",
        "medium": "Write approximately 150 words.",
        "detailed": "Write approximately 300 words."
    }

    if summary_length not in length_instructions:
        raise ValueError("Choose short, medium, or detailed.")

    # Retrieve relevant chunks
    results = search_documents(query, top_n=top_n)

    if results.empty:
        return "No relevant documents found.", results

    # Prepare source context
    context = "\n\n".join(
        f"[Document ID: {row.document_id}, "
        f"Chunk ID: {row.chunk_id}, "
        f"Category: {row.category}]\n{row.text}"
        for row in results.itertuples()
    )

    prompt = f"""
You are a document search and summarization assistant.

User question:
{query}

Retrieved source excerpts:
{context}

Instructions:
- Summarize the information relevant to the user's question.
- {length_instructions[summary_length]}
- Use only the supplied source excerpts.
- Do not invent facts or unsupported citations.
- Mention the relevant document IDs where appropriate.
- State when the sources do not provide enough information.
"""

    summary = generate_summary_with_gemini(prompt)

    return summary, results


print("Search and summarization pipeline is ready!")

Search and summarization pipeline is ready!


In [24]:
query = "machine learning algorithms and neural networks"

summary, sources = summarize_documents(
    query=query,
    summary_length="medium",
    top_n=5
)

print("USER QUERY:")
print(query)

print("\nGENERATED SUMMARY:")
print(summary)

print("\nSOURCE DOCUMENTS:")
display(
    sources[
        ["document_id", "chunk_id", "category", "hybrid_score", "text"]
    ]
)

USER QUERY:
machine learning algorithms and neural networks

GENERATED SUMMARY:
The provided sources discuss machine learning algorithms and neural networks, highlighting their applications and interconnections.

Machine learning techniques are applied to biological sequence analysis and are a topic within artificial intelligence (Document 16803). Genetic Algorithms and Genetic Programming are mentioned as a machine learning approach for non-linear multi-dimensional optimization problems, inspired by biology (Document 16803).

Neural networks are a key component of intelligent systems for molecular biology, used in biological sequence analysis to elicit signals and characteristics, and are related to statistical and information-theoretic views (Document 16803). They have shown significant progress in shape analysis, often incorporating energy minimization techniques like Markov random fields and simulated annealing (Document 15456). An annual book series reviews research on the modelin

,document_id,chunk_id,category,hybrid_score,text
0,16803,33465,sci.med,0.423891,problems in both academia and industry. 3:00-5...
1,15456,30746,comp.graphics,0.407818,CALL FOR PAPERS =============== Progress In Ne...
2,15456,30747,comp.graphics,0.386959,"topics include, but are not limited to: shape ..."
3,18032,35792,alt.atheism,0.377381,I'm sure there are many people who work with n...
4,16803,33464,sci.med,0.337983,"Science and Biology at Rutgers, and has extens..."


In [25]:

import numpy as np
import pandas as pd

# Fix random seed so results can be reproduced
rng = np.random.default_rng(42)

# Select up to 100 different chunks for evaluation
n_test = min(100, len(chunks_df))
test_indices = rng.choice(
    len(chunks_df),
    size=n_test,
    replace=False
)

test_set = chunks_df.iloc[test_indices].reset_index(drop=True)

# Use the first 12 words of each selected chunk as a query
test_set["query"] = test_set["text"].apply(
    lambda text: " ".join(text.split()[:12])
)

print("Number of evaluation queries:", len(test_set))
display(test_set[["document_id", "chunk_id", "category", "query"]].head())


Number of evaluation queries: 100


,document_id,chunk_id,category,query
0,7796,15660,misc.forsale,I have a video board for sale for Macintosh NU...
1,16922,33684,sci.crypt,250-<burrows@ecf> 250-<mcnulty@ecf> 250-Bill C...
2,11918,23994,sci.med,pimples but the cosmetic industry makes it's m...
3,8335,16706,comp.windows.x,I am looking for software to draw a graph. I w...
4,8021,16114,talk.politics.guns,You're getting warmer. The 'little thing in th...


In [26]:

def calculate_metrics(ranked_indices, relevant_index, k=5):
    top_k = ranked_indices[:k]

    # Exact matching chunk is the relevant target
    hits = [1 if idx == relevant_index else 0 for idx in top_k]

    precision = sum(hits) / k
    recall = float(relevant_index in top_k)

    if relevant_index in ranked_indices:
        rank = list(ranked_indices).index(relevant_index) + 1
        reciprocal_rank = 1 / rank
    else:
        reciprocal_rank = 0.0

    return precision, recall, reciprocal_rank


In [27]:

from sklearn.metrics.pairwise import cosine_similarity

def evaluate_search_methods(test_set, k=5):
    records = []

    for _, row in test_set.iterrows():
        query = row["query"]
        relevant_chunk = int(row["chunk_id"])

        # Find the row position corresponding to the target chunk
        target_positions = np.flatnonzero(
            chunks_df["chunk_id"].to_numpy() == relevant_chunk
        )
        target_index = int(target_positions[0])

        # 1. TF-IDF ranking
        q_tfidf = tfidf_vectorizer.transform(
            [normalize_text(query)]
        )
        tfidf_scores = (
            tfidf_matrix @ q_tfidf.T
        ).toarray().ravel()
        tfidf_ranked = np.argsort(tfidf_scores)[::-1]

        # 2. Semantic ranking
        q_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]
        semantic_scores = embeddings @ q_embedding
        semantic_ranked = np.argsort(semantic_scores)[::-1]

        # 3. Hybrid ranking
        hybrid_scores = (
            0.4 * tfidf_scores
            + 0.6 * semantic_scores
        )
        hybrid_ranked = np.argsort(hybrid_scores)[::-1]

        for method, ranked in [
            ("TF-IDF", tfidf_ranked),
            ("Semantic", semantic_ranked),
            ("Hybrid", hybrid_ranked)
        ]:
            p, r, mrr = calculate_metrics(
                ranked,
                target_index,
                k=k
            )

            records.append({
                "method": method,
                "precision_at_5": p,
                "recall_at_5": r,
                "reciprocal_rank": mrr
            })

    results_df = pd.DataFrame(records)

    return results_df.groupby("method").mean(numeric_only=True).reset_index()


search_evaluation = evaluate_search_methods(test_set)

print("Search Evaluation Results")
display(search_evaluation.round(4))


Search Evaluation Results


,method,precision_at_5,recall_at_5,reciprocal_rank
0,Hybrid,0.170,0.85,0.7409
1,Semantic,0.126,0.63,0.5417
2,TF-IDF,0.182,0.91,0.7266


In [28]:

%pip -q install rouge-score


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [29]:

from rouge_score import rouge_scorer

# Use a few queries to limit API usage
sample_size = min(5, len(test_set))
summary_test = test_set.head(sample_size)

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

summary_records = []

for _, row in summary_test.iterrows():
    query = row["query"]

    generated_summary, _ = summarize_documents(
        query=query,
        summary_length="short",
        top_n=3
    )

    # Reference text for this initial proxy evaluation
    reference = row["text"]

    scores = scorer.score(reference, generated_summary)

    summary_records.append({
        "query": query,
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure
    })

summary_evaluation = pd.DataFrame(summary_records)

print("ROUGE Evaluation Results")
display(summary_evaluation.round(4))

print("\nAverage ROUGE scores:")
display(
    summary_evaluation[["rouge1_f1", "rougeL_f1"]]
    .mean()
    .to_frame("average_score")
    .round(4)
)


ROUGE Evaluation Results


,query,rouge1_f1,rougeL_f1
0,I have a video board for sale for Macintosh NU...,0.4554,0.3564
1,250-<burrows@ecf> 250-<mcnulty@ecf> 250-Bill C...,0.1266,0.0570
2,pimples but the cosmetic industry makes it's m...,0.7397,0.4932
3,I am looking for software to draw a graph. I w...,0.3469,0.2041
4,You're getting warmer. The 'little thing in th...,0.3627,0.3212



Average ROUGE scores:


,average_score
rouge1_f1,0.4063
rougeL_f1,0.2864


In [30]:

import os

os.makedirs("/content/evaluation", exist_ok=True)

search_evaluation.to_csv(
    "/content/evaluation/search_evaluation.csv",
    index=False
)

summary_evaluation.to_csv(
    "/content/evaluation/summary_evaluation.csv",
    index=False
)

print("Evaluation results saved successfully!")


Evaluation results saved successfully!


In [31]:

# Report the actual ROUGE results

avg_rouge1 = summary_evaluation["rouge1_f1"].mean()
avg_rougeL = summary_evaluation["rougeL_f1"].mean()

report_results = f"""
LLM DOCUMENT SUMMARIZATION — EVALUATION

Evaluation method:
ROUGE-1 F1 and ROUGE-L F1

Number of evaluated summaries:
{len(summary_evaluation)}

Average ROUGE-1 F1:
{avg_rouge1:.4f}

Average ROUGE-L F1:
{avg_rougeL:.4f}

Interpretation:
The generated summaries show measurable lexical overlap
with the selected reference source texts. These values
are preliminary because the reference texts are original
document excerpts rather than human-written summaries.
"""
print(report_results)

with open("/content/evaluation/rouge_report.txt", "w") as f:
    f.write(report_results)

print("Report saved to /content/evaluation/rouge_report.txt")



LLM DOCUMENT SUMMARIZATION — EVALUATION

Evaluation method:
ROUGE-1 F1 and ROUGE-L F1

Number of evaluated summaries:
5

Average ROUGE-1 F1:
0.4063

Average ROUGE-L F1:
0.2864

Interpretation:
The generated summaries show measurable lexical overlap
with the selected reference source texts. These values
are preliminary because the reference texts are original
document excerpts rather than human-written summaries.

Report saved to /content/evaluation/rouge_report.txt


In [32]:
display(search_evaluation.round(4))

,method,precision_at_5,recall_at_5,reciprocal_rank
0,Hybrid,0.170,0.85,0.7409
1,Semantic,0.126,0.63,0.5417
2,TF-IDF,0.182,0.91,0.7266


In [33]:

%pip -q install streamlit pyngrok


In [34]:

%%writefile app.py
import streamlit as st
import pandas as pd

st.set_page_config(
    page_title="Document Search & Summarization",
    page_icon="📚",
    layout="wide"
)

st.title("📚 Document Search & Summarization")
st.write(
    "Search a document collection using hybrid retrieval "
    "and summarize relevant passages using an LLM."
)

# Search and summarization functions are supplied by the
# Colab notebook through a bridge file in the next step.
import pickle
import os

BRIDGE_FILE = "/content/document_app_bridge.pkl"

if not os.path.exists(BRIDGE_FILE):
    st.error("Application resources are not ready. Run the setup cell in Colab.")
    st.stop()

with open(BRIDGE_FILE, "rb") as f:
    resources = pickle.load(f)

search_documents = resources["search_documents"]
summarize_documents = resources["summarize_documents"]

query = st.text_input(
    "Enter your search query",
    placeholder="Example: machine learning algorithms"
)

summary_length = st.selectbox(
    "Choose summary length",
    ["short", "medium", "detailed"],
    index=1
)

top_n = st.slider(
    "Number of search results",
    min_value=1,
    max_value=10,
    value=5
)

if st.button("Search and Summarize", type="primary"):
    if not query.strip():
        st.warning("Please enter a query.")
    else:
        with st.spinner("Searching documents and generating summary..."):
            try:
                summary, results = summarize_documents(
                    query=query,
                    summary_length=summary_length,
                    top_n=top_n
                )

                st.subheader("Generated Summary")
                st.write(summary)

                st.subheader("Retrieved Documents")

                if results.empty:
                    st.info("No relevant documents were found.")
                else:
                    page_size = 3
                    total_pages = (len(results) - 1) // page_size + 1

                    page = st.number_input(
                        "Results page",
                        min_value=1,
                        max_value=total_pages,
                        value=1,
                        step=1
                    )

                    start = (page - 1) * page_size
                    page_results = results.iloc[start:start + page_size]

                    for _, result in page_results.iterrows():
                        with st.expander(
                            f"Document {result['document_id']} "
                            f"— {result['category']}"
                        ):
                            st.write(result["text"])

                            if "hybrid_score" in result:
                                st.write(
                                    "Hybrid score:",
                                    round(float(result["hybrid_score"]), 4)
                                )

            except Exception as e:
                st.error(f"Request failed: {e}")


Writing app.py


In [35]:

import pickle

resources = {
    "search_documents": search_documents,
    "summarize_documents": summarize_documents
}

with open("/content/document_app_bridge.pkl", "wb") as f:
    pickle.dump(resources, f)

print("Application resources saved.")


Application resources saved.


In [37]:

import ipywidgets as widgets
from IPython.display import display, clear_output

query_box = widgets.Text(
    description="Query:",
    placeholder="Enter your question",
    layout=widgets.Layout(width="80%")
)

length_dropdown = widgets.Dropdown(
    options=["short", "medium", "detailed"],
    value="medium",
    description="Length:"
)

top_n_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    description="Top N:"
)

run_button = widgets.Button(
    description="Search & Summarize",
    button_style="primary"
)

output = widgets.Output()

def run_search(button):
    with output:
        clear_output()

        if not query_box.value.strip():
            print("Please enter a query.")
            return

        try:
            summary, results = summarize_documents(
                query=query_box.value,
                summary_length=length_dropdown.value,
                top_n=top_n_slider.value
            )

            print("GENERATED SUMMARY\n")
            print(summary)

            print("\nRETRIEVED DOCUMENTS\n")
            display(results)

        except Exception as e:
            print("Error:", e)

run_button.on_click(run_search)

display(
    widgets.VBox([
        query_box,
        length_dropdown,
        top_n_slider,
        run_button,
        output
    ])
)


In [38]:
from pathlib import Path
import shutil

# Create the report folder inside your Google Colab environment
output_dir = Path("/content/final_project")
output_dir.mkdir(parents=True, exist_ok=True)

report = """
# Document Search and Summarization Using Large Language Models (LLM)

## 1. Project Overview
This project retrieves relevant text passages and uses a Large Language Model
(LLM) to generate a summary based on the retrieved information.

## 2. Objectives
- Load and preprocess a text dataset.
- Split documents into overlapping chunks.
- Implement keyword-based and semantic search.
- Combine both methods using hybrid search.
- Generate context-grounded summaries using Gemini.
- Evaluate retrieval and summarization performance.
- Build an interactive Google Colab interface.

## 3. Dataset
The project uses the 20 Newsgroups dataset from scikit-learn.
Headers, footers, and quoted text were excluded during dataset loading.

## 4. Preprocessing
The preprocessing pipeline removes empty and duplicate documents,
normalizes whitespace, creates normalized text, and divides documents
into chunks of approximately 150 words with a 30-word overlap.

## 5. Hybrid Search
Keyword retrieval uses TF-IDF with unigrams and bigrams.
Semantic retrieval uses sentence-transformers/all-MiniLM-L6-v2 embeddings.

The hybrid score is calculated as:

hybrid_score = 0.6 * semantic_score + 0.4 * keyword_score

The highest-ranked chunks are selected as context for the LLM.

## 6. LLM Summarization
The Gemini gemini-2.5-flash model is accessed through its REST API.
Retrieved document chunks are provided as context, and the prompt instructs
the model to avoid unsupported claims and identify source chunks where possible.

Users can choose short, medium, or detailed summaries.

## 7. Evaluation
The initial summarization evaluation produced:

- ROUGE-1 F1: 0.4063
- ROUGE-L F1: 0.2864

These are baseline results, not proof of factual accuracy. ROUGE measures
text overlap. The reference used was a source chunk rather than a
human-written summary, so the evaluation has limitations.

Search performance is evaluated separately using Precision@5, Recall@5,
and reciprocal rank/MRR where available.

## 8. Interactive Interface
The notebook interface allows users to enter a query, select a summary
length, choose the number of retrieved chunks, and view the summary
alongside retrieved results.

## 9. Limitations
- Search evaluation depends on how relevant documents are defined.
- Queries derived from source chunks may produce an optimistic baseline.
- ROUGE does not measure factual correctness.
- LLM-generated summaries may contain errors and should be checked
  against the retrieved passages.
- Gemini API access requires a valid API key and is subject to usage limits.

## 10. Future Improvements
- Evaluate with real user queries and manually labelled relevant passages.
- Use human-written reference summaries.
- Add persistent vector storage and document-upload support.
- Improve source attribution and factual-grounding checks.
- Test the system on domain-specific documents.

## 11. Conclusion
This project combines text preprocessing, TF-IDF, semantic embeddings,
hybrid retrieval, and LLM-based summarization in an interactive workflow.
It demonstrates how relevant passages can be retrieved and summarized,
while providing baseline evaluation results for future improvement.
"""

report_path = output_dir / "project_report.md"
report_path.write_text(report.strip(), encoding="utf-8")

# Copy available project data and evaluation results
files_to_copy = [
    "/content/data/cleaned_documents.csv",
    "/content/data/document_chunks.csv",
    "/content/evaluation/search_evaluation.csv",
    "/content/evaluation/summary_evaluation.csv",
    "/content/evaluation/rouge_report.txt",
]

for file_path in files_to_copy:
    source = Path(file_path)
    if source.exists():
        shutil.copy2(source, output_dir / source.name)
        print("Copied:", source.name)
    else:
        print("Not found, skipped:", source.name)

print("\nReport saved to:", report_path)
print("Step 7 completed.")

Copied: cleaned_documents.csv
Copied: document_chunks.csv
Copied: search_evaluation.csv
Copied: summary_evaluation.csv
Copied: rouge_report.txt

Report saved to: /content/final_project/project_report.md
Step 7 completed.


In [39]:
import os

for filename in os.listdir("/content/final_project"):
    print(filename)

rouge_report.txt
summary_evaluation.csv
cleaned_documents.csv
project_report.md
search_evaluation.csv
document_chunks.csv


In [40]:
import os
import zipfile

folder_path = "/content/final_project"
zip_path = "/content/Document_Search_Summarization_Project.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        if os.path.isfile(file_path):
            zipf.write(file_path, arcname=filename)

print("ZIP file created successfully!")
print("Saved at:", zip_path)
print("Files included:")

with zipfile.ZipFile(zip_path, "r") as zipf:
    for filename in zipf.namelist():
        print("-", filename)

ZIP file created successfully!
Saved at: /content/Document_Search_Summarization_Project.zip
Files included:
- rouge_report.txt
- summary_evaluation.csv
- cleaned_documents.csv
- project_report.md
- search_evaluation.csv
- document_chunks.csv


In [41]:
from google.colab import files

files.download("/content/Document_Search_Summarization_Project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
from pathlib import Path

readme = """
# Document Search and Summarization Using LLMs

## Project Overview
An intelligent document search and summarization system that retrieves
relevant passages from a text dataset and generates summaries using a
Large Language Model (LLM).

## Technologies Used
- Python
- Pandas and NumPy
- Scikit-learn
- TF-IDF and cosine similarity
- Sentence Transformers
- Gemini 2.5 Flash API
- Google Colab
- IPyWidgets

## Dataset
The project uses the 20 Newsgroups dataset from scikit-learn.

## Key Features
- Text cleaning and preprocessing
- Document chunking with overlapping text
- Keyword-based retrieval using TF-IDF
- Semantic retrieval using sentence embeddings
- Hybrid search combining keyword and semantic scores
- LLM-powered summarization
- Adjustable summary length and retrieval count
- Search and summarization evaluation

## Evaluation Results
- ROUGE-1 F1: 0.4063
- ROUGE-L F1: 0.2864

These are initial baseline results. ROUGE measures text overlap and
does not independently establish factual accuracy.

## How It Works
1. Load and preprocess the documents.
2. Split documents into overlapping chunks.
3. Retrieve relevant chunks using hybrid search.
4. Send retrieved context to the Gemini model.
5. Display the generated summary and source passages.
6. Evaluate retrieval and summarization performance.

## Important
A valid Gemini API key is required to generate summaries.
Do not publish your API key in the notebook or repository.

## Future Improvements
- Evaluate with real user queries and labelled relevant documents.
- Add document uploads and persistent vector storage.
- Improve source attribution and factual-grounding checks.
"""

Path("/content/final_project/README.md").write_text(
    readme.strip(), encoding="utf-8"
)

print("README.md created successfully!")

README.md created successfully!
